# Thyroid Cancer Recurrence: Random Forest Classification
**Author:** Justin Williams  
**Date:** 11/3/2026

A demonstration of the Random Forest classification model. The notebook compares two evaluation methods: train/test splits and 5-fold cross-validation.

## Contents

1. Background
2. Setup and Constants
3. Load Data
4. Preprocessing
5. Model Pipeline
6. Evaluation
7. Results


## 1. Background

A `RandomForestClassifier` is an ensemble of decision trees. Each tree is trained on a random sample of rows and considers a random selection of features at each split. Each tree votes, and the majority vote is chosen for the final prediction.



## 2. Setup and Constants

In [163]:
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn import preprocessing
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split
)

In [164]:
TARGET_COL = "Recurred"
CATEGORICAL_COLS = [
    "Gender",
    "Smoking",
    "Hx Smoking",
    "Hx Radiotherapy",
    "Thyroid Function",
    "Physical Examination",
    "Adenopathy",
    "Pathology",
    "Focality",
    "Risk",
    "T",
    "N",
    "M",
    "Stage",
    "Response"
]

RAND_STATE = 42

TEST_SIZES = [0.2, 0.25, 0.3]

PARAM_DICT = {
    "clf__max_depth": [10, 20, 30],
    "clf__max_features": ["sqrt", "log2"],
    "clf__n_estimators": [15, 30, 75, 90,]
}

N_COMPONENTS = 21

pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", '{:.3f}'.format)

In [165]:
# Format a confusion matrix (cm) into a Pandas dataframe
def make_cm_frame(cm):
    return pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Pred No", "Pred Yes"]
    )


### Cross-Validation Setup

`StratifiedKFold` keeps the ratio of target values consistent across each fold.

In [166]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_STATE)

### SMOTE Setup

Only 28.2% of patients recurred. SMOTE ensures that the target data is not imbalanced during training.

In [167]:
smote = SMOTE(sampling_strategy="minority", random_state=RAND_STATE)

## 3. Load Data

In [168]:
df = pd.read_csv("../data/processed/Thyroid_Diff.csv")

print(df.shape)

df.head()

(383, 17)


,Age,Gender,Smoking,Hx Smoking,Hx Radiotherapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


In [169]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Age                   383 non-null    int64
 1   Gender                383 non-null    str  
 2   Smoking               383 non-null    str  
 3   Hx Smoking            383 non-null    str  
 4   Hx Radiotherapy       383 non-null    str  
 5   Thyroid Function      383 non-null    str  
 6   Physical Examination  383 non-null    str  
 7   Adenopathy            383 non-null    str  
 8   Pathology             383 non-null    str  
 9   Focality              383 non-null    str  
 10  Risk                  383 non-null    str  
 11  T                     383 non-null    str  
 12  N                     383 non-null    str  
 13  M                     383 non-null    str  
 14  Stage                 383 non-null    str  
 15  Response              383 non-null    str  
 16  Recurred           

## 4. Preprocessing

`X` holds the features to train on. `y` holds the target to predict. The target column maps Yes/No to 1/0.

In [170]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1})

print(y.value_counts())
print(f"\nRecurrence rate: {y.mean():.1%}")

Recurred
0    275
1    108
Name: count, dtype: int64

Recurrence rate: 28.2%


### Encoding and Scaling

Columns listed in `CATEGORICAL_COLS` are transformed using One-Hot Encoding. The `Age` column is scaled to have a mean of 0 and a standard deviation of 1. The `ColumnTransformer` is used to avoid data leakage from encoding training and testing data together. It produces the same columns, even if future data is missing any categories.


In [171]:
colTransform = ColumnTransformer(
    transformers=[
        ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", preprocessing.StandardScaler(), ["Age"])
    ],
    verbose_feature_names_out=False
)

## 5. Pipeline Setup

In [172]:
model = Pipeline([
    ("trans", colTransform),
    ("smote", smote),
    ("pca", PCA(n_components=N_COMPONENTS, random_state=RAND_STATE)),
    ("clf", RandomForestClassifier(random_state=RAND_STATE))
])

model

,steps,"[('trans', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weight

### Hyperparameter Tuning - GridSearchCV

Adjust the parameters of the Random Forest classifier in the pipeline to find the combination of parameters with the best results. Fit to each train/test split.

In [173]:
gs = GridSearchCV(model, PARAM_DICT)

## 6. Evaluation

### Train-Test Splits

In [174]:
results = []

for size in TEST_SIZES:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=size, stratify=y, random_state=RAND_STATE
    )

    gs = GridSearchCV(model, PARAM_DICT)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)

    enc = gs.best_estimator_.named_steps["trans"]
    pca = gs.best_estimator_.named_steps["pca"]
    loadings = pd.DataFrame(
        (pca.components_.T * np.sqrt(pca.explained_variance_)),
        index=enc.get_feature_names_out(),
        columns=(f"PC{i+1}" for i in range(pca.n_components_))
    )

    results.append({
        "Method": f"Train/test split ({size})",
        "Report": classification_report(y_test, y_pred, target_names=["No", "Yes"], output_dict=True),
        "CM": confusion_matrix(y_test, y_pred),
        "ROCAUC": roc_auc,
        "Best": gs.best_params_,
        "FeatLoad": loadings
    })

### 5-Fold Cross-Validation

In [175]:
folds = []

for i_train, i_test in kfold.split(X, y):
    X_train, X_test, y_train, y_test = X.iloc[i_train], X.iloc[i_test], y.iloc[i_train], y.iloc[i_test]
    gs.fit(X_train, y_train)

    enc = gs.best_estimator_.named_steps["trans"]
    pca = gs.best_estimator_.named_steps["pca"]
    loadings = pd.DataFrame(
        (pca.components_.T * np.sqrt(pca.explained_variance_)),
        index=enc.get_feature_names_out(),
        columns=(f"PC{i+1}" for i in range(pca.n_components_))
    )

    folds.append({
        "i_train": i_train,
        "i_test": i_test,
        "acc": gs.score(X_test, y_test),
        "pred": gs.predict(X_test),
        "test": y_test,
        "params": gs.best_params_,
        "loadings": loadings
    })

fold_scores = [f.get("acc") for f in folds]

high = max(fold_scores)
high_fold = folds[fold_scores.index(high)]
low = min(fold_scores)
low_fold = folds[fold_scores.index(low)]
avg = np.mean(fold_scores)

cv_cm = confusion_matrix(high_fold["test"], high_fold["pred"])

fpr, tpr, thresholds = roc_curve(high_fold["test"], high_fold["pred"])
roc_auc = auc(fpr, tpr)
    
results.append({
    "Method": "5-fold cross-validation (max)",
    "Report": classification_report(high_fold["test"], high_fold["pred"], target_names=["No", "Yes"], output_dict=True),
    "CM": cv_cm,
    "ROCAUC": roc_auc,
    "Best": high_fold["params"],
    "FeatLoad": high_fold["loadings"]
})

cv_cm = confusion_matrix(low_fold["test"], low_fold["pred"])

fpr, tpr, thresholds = roc_curve(low_fold["test"], low_fold["pred"])
roc_auc = auc(fpr, tpr)

results.append({
    "Method": "5-fold cross-validation (min)",
    "Report": classification_report(low_fold["test"], low_fold["pred"], target_names=["No", "Yes"], output_dict=True),
    "CM": cv_cm,
    "ROCAUC": roc_auc,
    "Best": low_fold["params"],
    "FeatLoad": low_fold["loadings"]
})

# Compute average classification report and average ROC AUC statistics
cv_report_list = []
roc_auc_list = []
for f in folds:
    cv_report_list.append(pd.DataFrame(classification_report(f["test"], f["pred"], target_names=["No", "Yes"], output_dict=True)))
    fpr, tpr, thresholds = roc_curve(f["test"], f["pred"])
    roc_auc_list.append(auc(fpr, tpr))

cv_avg_report = pd.concat(cv_report_list).groupby(level=0).mean()
cv_avg_auc = np.mean(roc_auc_list)


In [176]:
print(f"KFold Scores: {fold_scores}")
print(f"KFold Average: {avg}")

KFold Scores: [0.948051948051948, 0.935064935064935, 0.948051948051948, 0.9078947368421053, 0.9078947368421053]
KFold Average: 0.9293916609706084


## 7. Results

### Classification Reports

In [177]:
for result in results:
    print(result["Method"])
    print(f"ROC AUC: {result["ROCAUC"]}")
    display(pd.DataFrame(result["Report"]))

print("5-fold cross-validation (avg)")
print(cv_avg_auc)
display(cv_avg_report)


Train/test split (0.2)
ROC AUC: 0.9454545454545454


,No,Yes,accuracy,macro avg,weighted avg
precision,0.964,0.952,0.961,0.958,0.961
recall,0.982,0.909,0.961,0.945,0.961
f1-score,0.973,0.930,0.961,0.952,0.961
support,55.000,22.000,0.961,77.000,77.000


Train/test split (0.25)
ROC AUC: 0.9484702093397746


,No,Yes,accuracy,macro avg,weighted avg
precision,0.971,0.926,0.958,0.948,0.958
recall,0.971,0.926,0.958,0.948,0.958
f1-score,0.971,0.926,0.958,0.948,0.958
support,69.000,27.000,0.958,96.000,96.000


Train/test split (0.3)
ROC AUC: 0.9290286144578314


,No,Yes,accuracy,macro avg,weighted avg
precision,0.963,0.879,0.939,0.921,0.940
recall,0.952,0.906,0.939,0.929,0.939
f1-score,0.958,0.892,0.939,0.925,0.939
support,83.000,32.000,0.939,115.000,115.000


... (2 outputs left out)

5-fold cross-validation (min)
ROC AUC: 0.9216450216450216


,No,Yes,accuracy,macro avg,weighted avg
precision,0.980,0.769,0.908,0.875,0.922
recall,0.891,0.952,0.908,0.922,0.908
f1-score,0.933,0.851,0.908,0.892,0.911
support,55.000,21.000,0.908,76.000,76.000


5-fold cross-validation (avg)
0.9198268398268399


,No,Yes,accuracy,macro avg,weighted avg
f1-score,0.950,0.877,0.929,0.914,0.930
precision,0.960,0.861,0.929,0.911,0.932
recall,0.942,0.898,0.929,0.920,0.929
support,55.000,21.600,0.929,76.600,76.600


### Confusion Matrices

The rows represent the actual classes, and the columns represent predicted classes. All correct predictions lie on the diagonal:

| | Pred No | Pred Yes |
|---|---|---|
| **Actual No** | True negatives (TN) | False positives (FP) |
| **Actual Yes** | False negatives (FN) | True positives (TP) |

Train/Test Split matrices count the test set while the Cross-Validation matrix counts the entire dataset.

In [178]:
for result in results:
    print(result["Method"])
    display(make_cm_frame(result["CM"]))


Train/test split (0.2)


,Pred No,Pred Yes
Actual No,54,1
Actual Yes,2,20


Train/test split (0.25)


,Pred No,Pred Yes
Actual No,67,2
Actual Yes,2,25


Train/test split (0.3)


,Pred No,Pred Yes
Actual No,79,4
Actual Yes,3,29


5-fold cross-validation (max)


,Pred No,Pred Yes
Actual No,53,2
Actual Yes,2,20


5-fold cross-validation (min)


,Pred No,Pred Yes
Actual No,49,6
Actual Yes,1,20


### Hyperparameter Tuning - GridSearchCV

The following hyper parameters for the Random Forest model were tuned:
1. `max_depth` - How deep each tree can go
2. `max_features` - Total number of features each tree can train on
3. `n_estimators` - Number of trees voting


In [179]:
for result in results:
    print(result["Method"])
    print(result["Best"])
    print()

Train/test split (0.2)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 30}

Train/test split (0.25)
{'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__n_estimators': 90}

Train/test split (0.3)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 30}

5-fold cross-validation (max)
{'clf__max_depth': 20, 'clf__max_features': 'sqrt', 'clf__n_estimators': 30}

5-fold cross-validation (min)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 15}



### Feature Loadings

Matrix of feature loadings, explaining the impact of each feature across each component.


In [180]:
for result in results:
    print(result["Method"])
    display(result["FeatLoad"])

Train/test split (0.2)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Gender_F,-0.205,0.009,-0.252,0.035,-0.132,-0.170,-0.025,0.005,-0.010,-0.005,...,0.025,0.087,0.025,-0.028,-0.036,0.025,0.002,-0.063,-0.013,-0.008
Gender_M,0.205,-0.009,0.252,-0.035,0.132,0.170,0.025,-0.005,0.010,0.005,...,-0.025,-0.087,-0.025,0.028,0.036,-0.025,-0.002,0.063,0.013,0.008
Smoking_No,-0.210,-0.053,-0.212,0.044,-0.074,-0.063,0.001,-0.005,-0.016,0.004,...,0.020,-0.084,-0.080,-0.018,0.050,-0.024,0.028,0.091,0.024,0.002
Smoking_Yes,0.210,0.053,0.212,-0.044,0.074,0.063,-0.001,0.005,0.016,-0.004,...,-0.020,0.084,0.080,0.018,-0.050,0.024,-0.028,-0.091,-0.024,-0.002
Hx Smoking_No,-0.047,-0.051,-0.062,-0.002,0.013,-0.012,0.029,0.055,0.029,0.003,...,-0.102,-0.036,-0.014,0.017,-0.023,0.047,-0.137,-0.005,0.013,-0.011
Hx Smoking_Yes,0.047,0.051,0.062,0.002,-0.013,0.012,-0.029,-0.055,-0.029,-0.003,...,0.102,0.036,0.014,-0.017,0.023,-0.047,0.137,0.005,-0.013,0.011
Hx Radiotherapy_No,-0.022,-0.024,-0.031,-0.004,0.005,0.018,-0.010,0.002,0.021,-0.015,...,-0.010,0.007,-0.019,0.002,0.011,0.008,-0.007,-0.038,0.021,0.023
Hx Radiotherapy_Yes,0.022,0.024,0.031,0.004,-0.005,-0.018,0.010,-0.002,-0.021,0.015,...,0.010,-0.007,0.019,-0.002,-0.011,-0.008,0.007,0.038,-0.021,-0.023
Thyroid Function_Clinical Hyperthyroidism,-0.009,0.025,0.017,0.024,0.003,-0.012,-0.013,-0.021,-0.007,0.005,...,0.009,0.001,-0.007,-0.025,0.028,0.024,-0.047,0.043,-0.055,-0.028
Thyroid Function_Clinical Hypothyroidism,-0.015,0.006,-0.001,0.006,-0.005,0.012,0.004,-0.006,0.001,0.007,...,0.012,0.004,-0.020,-0.017,0.010,0.010,-0.040,0.002,-0.014,-0.025


Train/test split (0.25)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Gender_F,-0.194,-0.002,0.188,0.118,0.136,-0.158,-0.094,-0.036,0.016,0.009,...,-0.080,-0.053,-0.005,-0.014,-0.067,-0.042,-0.019,0.002,-0.002,0.018
Gender_M,0.194,0.002,-0.188,-0.118,-0.136,0.158,0.094,0.036,-0.016,-0.009,...,0.080,0.053,0.005,0.014,0.067,0.042,0.019,-0.002,0.002,-0.018
Smoking_No,-0.210,-0.069,0.184,0.085,0.077,-0.053,-0.033,-0.022,0.005,0.036,...,0.053,0.087,-0.061,-0.030,0.079,0.061,0.021,-0.038,0.007,-0.024
Smoking_Yes,0.210,0.069,-0.184,-0.085,-0.077,0.053,0.033,0.022,-0.005,-0.036,...,-0.053,-0.087,0.061,0.030,-0.079,-0.061,-0.021,0.038,-0.007,0.024
Hx Smoking_No,-0.049,-0.053,0.053,0.031,-0.011,0.024,-0.045,-0.025,0.042,0.051,...,0.108,-0.045,0.007,0.013,-0.007,0.025,-0.093,0.102,0.010,-0.011
Hx Smoking_Yes,0.049,0.053,-0.053,-0.031,0.011,-0.024,0.045,0.025,-0.042,-0.051,...,-0.108,0.045,-0.007,-0.013,0.007,-0.025,0.093,-0.102,-0.010,0.011
Hx Radiotherapy_No,-0.033,-0.029,0.030,0.034,0.002,0.010,0.016,-0.009,0.016,0.010,...,0.027,-0.011,-0.008,0.010,-0.002,-0.045,-0.015,0.010,0.032,0.024
Hx Radiotherapy_Yes,0.033,0.029,-0.030,-0.034,-0.002,-0.010,-0.016,0.009,-0.016,-0.010,...,-0.027,0.011,0.008,-0.010,0.002,0.045,0.015,-0.010,-0.032,-0.024
Thyroid Function_Clinical Hyperthyroidism,-0.004,0.027,-0.007,-0.035,-0.008,-0.026,0.019,0.010,-0.016,-0.068,...,-0.046,0.002,-0.026,-0.027,0.039,0.039,-0.052,0.010,-0.056,-0.017
Thyroid Function_Clinical Hypothyroidism,-0.017,0.006,0.003,-0.001,0.003,0.008,0.005,0.009,-0.009,-0.043,...,-0.028,0.002,-0.033,-0.013,0.019,0.004,-0.027,0.029,0.015,-0.003


Train/test split (0.3)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Gender_F,-0.188,-0.009,0.187,0.024,-0.001,-0.240,-0.046,-0.064,0.019,0.048,...,0.061,-0.073,0.013,0.003,-0.047,0.028,-0.025,-0.006,0.001,-0.019
Gender_M,0.188,0.009,-0.187,-0.024,0.001,0.240,0.046,0.064,-0.019,-0.048,...,-0.061,0.073,-0.013,-0.003,0.047,-0.028,0.025,0.006,-0.001,0.019
Smoking_No,-0.189,-0.055,0.186,0.057,0.042,-0.090,-0.034,-0.035,0.011,0.008,...,-0.055,0.076,-0.018,-0.083,0.075,-0.047,0.021,0.047,-0.029,0.019
Smoking_Yes,0.189,0.055,-0.186,-0.057,-0.042,0.090,0.034,0.035,-0.011,-0.008,...,0.055,-0.076,0.018,0.083,-0.075,0.047,-0.021,-0.047,0.029,-0.019
Hx Smoking_No,-0.038,-0.050,0.065,-0.001,0.019,-0.018,0.021,0.044,0.079,0.000,...,0.028,0.063,0.008,0.003,-0.007,0.093,0.102,-0.065,-0.028,-0.034
Hx Smoking_Yes,0.038,0.050,-0.065,0.001,-0.019,0.018,-0.021,-0.044,-0.079,-0.000,...,-0.028,-0.063,-0.008,-0.003,0.007,-0.093,-0.102,0.065,0.028,0.034
Hx Radiotherapy_No,-0.038,-0.033,0.055,0.003,0.009,-0.001,-0.006,0.006,0.030,0.001,...,-0.002,0.017,0.005,-0.006,-0.010,0.036,-0.016,-0.002,0.065,0.008
Hx Radiotherapy_Yes,0.038,0.033,-0.055,-0.003,-0.009,0.001,0.006,-0.006,-0.030,-0.001,...,0.002,-0.017,-0.005,0.006,0.010,-0.036,0.016,0.002,-0.065,-0.008
Thyroid Function_Clinical Hyperthyroidism,0.001,0.032,-0.039,0.014,-0.023,0.006,-0.020,-0.022,-0.038,-0.056,...,0.012,-0.061,-0.003,-0.025,0.044,0.016,0.029,0.013,-0.061,-0.014
Thyroid Function_Clinical Hypothyroidism,-0.016,0.006,-0.002,0.007,0.005,0.010,0.007,-0.010,-0.020,-0.033,...,0.016,-0.033,0.001,-0.035,0.018,0.032,0.011,-0.022,0.033,-0.033


5-fold cross-validation (max)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Gender_F,-0.214,0.006,-0.290,0.071,-0.171,0.019,0.024,-0.053,-0.046,-0.013,...,-0.002,-0.070,-0.006,0.023,-0.017,0.043,-0.005,0.039,0.066,-0.015
Gender_M,0.214,-0.006,0.290,-0.071,0.171,-0.019,-0.024,0.053,0.046,0.013,...,0.002,0.070,0.006,-0.023,0.017,-0.043,0.005,-0.039,-0.066,0.015
Smoking_No,-0.186,-0.101,-0.190,0.075,-0.077,0.068,0.003,-0.061,-0.009,0.031,...,-0.023,0.063,-0.008,-0.059,-0.025,-0.038,0.042,-0.057,-0.088,0.008
Smoking_Yes,0.186,0.101,0.190,-0.075,0.077,-0.068,-0.003,0.061,0.009,-0.031,...,0.023,-0.063,0.008,0.059,0.025,0.038,-0.042,0.057,0.088,-0.008
Hx Smoking_No,-0.051,-0.038,-0.071,0.024,-0.025,0.014,0.069,0.046,0.101,0.031,...,0.072,0.105,0.078,-0.006,0.096,0.029,-0.062,0.021,0.025,0.000
Hx Smoking_Yes,0.051,0.038,0.071,-0.024,0.025,-0.014,-0.069,-0.046,-0.101,-0.031,...,-0.072,-0.105,-0.078,0.006,-0.096,-0.029,0.062,-0.021,-0.025,-0.000
Hx Radiotherapy_No,-0.028,-0.032,-0.025,0.022,0.014,0.015,0.013,-0.001,0.028,0.002,...,0.002,0.019,0.018,0.001,-0.024,-0.011,0.004,0.043,-0.000,0.021
Hx Radiotherapy_Yes,0.028,0.032,0.025,-0.022,-0.014,-0.015,-0.013,0.001,-0.028,-0.002,...,-0.002,-0.019,-0.018,-0.001,0.024,0.011,-0.004,-0.043,0.000,-0.021
Thyroid Function_Clinical Hyperthyroidism,-0.014,0.026,0.014,0.013,-0.002,-0.020,-0.023,-0.027,-0.025,-0.038,...,0.015,0.015,-0.011,-0.067,0.037,0.051,0.016,-0.032,0.026,-0.002
Thyroid Function_Clinical Hypothyroidism,-0.012,0.016,-0.000,0.003,0.006,0.008,-0.009,0.010,-0.010,-0.022,...,0.004,0.016,-0.028,-0.035,0.020,0.008,0.017,0.017,0.018,-0.002


5-fold cross-validation (min)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Gender_F,-0.221,-0.053,0.116,0.238,0.005,0.176,-0.075,0.011,0.018,0.007,...,0.087,0.008,-0.060,-0.038,0.061,0.030,0.014,-0.018,0.029,0.004
Gender_M,0.221,0.053,-0.116,-0.238,-0.005,-0.176,0.075,-0.011,-0.018,-0.007,...,-0.087,-0.008,0.060,0.038,-0.061,-0.030,-0.014,0.018,-0.029,-0.004
Smoking_No,-0.188,-0.117,0.135,0.183,0.057,0.056,-0.025,-0.003,0.010,0.020,...,-0.076,-0.059,0.018,0.024,-0.095,-0.022,-0.013,-0.000,-0.079,0.019
Smoking_Yes,0.188,0.117,-0.135,-0.183,-0.057,-0.056,0.025,0.003,-0.010,-0.020,...,0.076,0.059,-0.018,-0.024,0.095,0.022,0.013,0.000,0.079,-0.019
Hx Smoking_No,-0.061,-0.059,0.050,0.071,-0.047,0.022,0.032,-0.030,0.013,0.029,...,-0.128,0.080,0.053,0.008,0.052,-0.082,-0.062,-0.027,0.048,0.023
Hx Smoking_Yes,0.061,0.059,-0.050,-0.071,0.047,-0.022,-0.032,0.030,-0.013,-0.029,...,0.128,-0.080,-0.053,-0.008,-0.052,0.082,0.062,0.027,-0.048,-0.023
Hx Radiotherapy_No,-0.036,-0.042,0.037,0.045,0.009,-0.031,-0.002,-0.013,0.020,0.002,...,-0.025,0.013,0.020,-0.003,0.037,0.001,0.029,0.047,0.020,0.011
Hx Radiotherapy_Yes,0.036,0.042,-0.037,-0.045,-0.009,0.031,0.002,0.013,-0.020,-0.002,...,0.025,-0.013,-0.020,0.003,-0.037,-0.001,-0.029,-0.047,-0.020,-0.011
Thyroid Function_Clinical Hyperthyroidism,-0.013,0.030,0.012,-0.030,0.012,-0.002,-0.032,0.022,-0.030,-0.072,...,0.020,-0.067,-0.009,0.013,0.005,-0.089,-0.014,-0.056,0.012,-0.033
Thyroid Function_Clinical Hypothyroidism,-0.019,0.006,0.002,0.006,0.013,-0.003,-0.001,0.008,-0.007,-0.026,...,-0.002,-0.030,-0.005,-0.008,0.010,-0.028,-0.003,-0.005,0.024,0.018


### PCA Results

The following table shows the accuracies produced by each train/test split for a series of PCA values. The highest accuracy for each split is in bold.

| Components | 0.2 split | 0.25 split | 0.3 split | 5-fold CV (avg) |
| ---------- | --------- | ---------- | --------- | --------------- |
| 17         | 92.2%     | **95.8%**  | **94.8%** | 91.9%           |
| 18         | 93.5%     | **95.8%**  | **94.8%** | 93.5%           |
| 19         | 93.5%     | **95.8%**  | **94.8%** | **93.7%**       |
| 20         | 93.5%     | 92.7%      | 91.3%     | 92.2%           |
| 21         | **96.1%** | **95.8%**  | 93.9%     | 92.9%           |
| 22         | 94.8%     | 94.8%      | 93.9%     | 92.7%           |
| 23         | 93.5%     | 89.6%      | 93.9%     | 93.5%           |
